In [7]:
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [8]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

In [9]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_generator = train_datagen.flow_from_directory(
    "../dataset/data",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training"
)

validation_generator = train_datagen.flow_from_directory(
    "../dataset/data",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation"
)

Found 5954 images belonging to 2 classes.
Found 1488 images belonging to 2 classes.


In [10]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

In [11]:
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(128, activation="relu")(x)

x = Dropout(0.5)(x)

output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

In [12]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [13]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=20,
    callbacks=[early_stop]
)

Epoch 1/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 696s 4s/step - accuracy: 0.8962 - loss: 0.2437 - val_accuracy: 0.9624 - val_loss: 0.0989
Epoch 2/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 521s 3s/step - accuracy: 0.9458 - loss: 0.1414 - val_accuracy: 0.9664 - val_loss: 0.0824
Epoch 3/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 519s 3s/step - accuracy: 0.9590 - loss: 0.1048 - val_accuracy: 0.9751 - val_loss: 0.0642
Epoch 4/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 505s 3s/step - accuracy: 0.9538 - loss: 0.1082 - val_accuracy: 0.9657 - val_loss: 0.0841
Epoch 5/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 504s 3s/step - accuracy: 0.9694 - loss: 0.0821 - val_accuracy: 0.9738 - val_loss: 0.0595
Epoch 6/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 501s 3s/step - accuracy: 0.9720 - loss: 0.0746 - val_accuracy: 0.9637 - val_loss: 0.0787
Epoch 7/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 488s 3s/step - accuracy: 0.9735 - loss: 0.0700 - val_accuracy: 0.9637 - val_loss: 0.0776
Epoch 8/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 492s 3s/step - accuracy: 0.9740 - loss: 0.0689 - val_accu

In [14]:
model.save("../models/mobilenet_counterfeit_detector.keras")